سلول ۱ — Import و مسیرها

In [ ]:
from pathlib import Path
import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)

from tqdm.auto import tqdm

سلول ۲ — تنظیمات اصلی

In [ ]:
PROJECT_ROOT = Path(".")

FUSION_DIR = (
    PROJECT_ROOT
    / "Data_ml"
    / "cross_model_fusion_results"
    / "esm650__esm3b__protbert_bfd"
)

X_PATH = FUSION_DIR / "cross_model_fusion.features.parquet"
META_PATH = FUSION_DIR / "meta.csv"

RESULT_DIR = PROJECT_ROOT / "Data_ml" / "neural_pair_scorer_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_SPLITS = 5

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)
print("X exists:", X_PATH.exists())
print("META exists:", META_PATH.exists())

سلول ۳ — Load data

In [ ]:
X_df = pd.read_parquet(X_PATH)
meta = pd.read_csv(META_PATH, dtype=str, low_memory=False)

y = meta["label"].astype(int).values
groups = meta["group_id"].values

print("X:", X_df.shape)
print("meta:", meta.shape)
print("label counts:")
print(pd.Series(y).value_counts())

display(X_df.head())
display(meta.head())

سلول ۴ — Dataset

In [ ]:
class PairFeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

سلول ۵ — معماری مدل

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.25):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.net(x)


class NeuralPairScorer(nn.Module):
    def __init__(
        self,
        input_dim,
        projection_dim=1024,
        block_hidden_dim=2048,
        n_blocks=3,
        dropout=0.25,
    ):
        super().__init__()

        self.input = nn.Sequential(
            nn.Linear(input_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.blocks = nn.Sequential(
            *[
                ResidualBlock(
                    dim=projection_dim,
                    hidden_dim=block_hidden_dim,
                    dropout=dropout,
                )
                for _ in range(n_blocks)
            ]
        )

        self.head = nn.Sequential(
            nn.LayerNorm(projection_dim),
            nn.Linear(projection_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.input(x)
        x = self.blocks(x)
        logit = self.head(x).squeeze(-1)
        return logit

سلول ۶ — تابع متریک

In [ ]:
def compute_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)

    pred = (prob >= threshold).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "f1": f1_score(y_true, pred, zero_division=0),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "brier": brier_score_loss(y_true, prob),
    }

سلول ۷ — Training loop با Early Stopping

In [ ]:
def train_one_fold(
    X_train,
    y_train,
    X_val,
    y_val,
    input_dim,
    loss_name="bce",
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
    patience=12,
    model_config=None,
):
    if model_config is None:
        model_config = {}

    train_ds = PairFeatureDataset(X_train, y_train)
    val_ds = PairFeatureDataset(X_val, y_val)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    model = NeuralPairScorer(
        input_dim=input_dim,
        **model_config,
    ).to(DEVICE)

    if loss_name == "bce":
        criterion = nn.BCEWithLogitsLoss()
    else:
        raise ValueError(loss_name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
    )

    best_pr = -np.inf
    best_state = None
    best_epoch = -1
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_probs = []
        val_true = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)

                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()

                val_probs.extend(prob.tolist())
                val_true.extend(yb.numpy().tolist())

        val_metrics = compute_metrics(val_true, val_probs)
        scheduler.step(val_metrics["pr_auc"])

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            **val_metrics,
        }

        history.append(row)

        if val_metrics["pr_auc"] > best_pr:
            best_pr = val_metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 5 == 0:
            print(
                f"epoch={epoch:03d} "
                f"loss={row['train_loss']:.4f} "
                f"val_pr={row['pr_auc']:.4f} "
                f"val_roc={row['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping at epoch:", epoch)
            break

    model.load_state_dict(best_state)
    model = model.to(DEVICE)
    model.eval()

    return model, pd.DataFrame(history), best_epoch, best_pr

سلول ۸ — Cross-validation برای NN

In [ ]:
def run_nn_cv(
    X_df,
    y,
    groups,
    loss_name="bce",
    model_name="nn_bce",
    model_config=None,
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
):
    X_np = X_df.to_numpy(dtype=np.float32)
    y_np = np.asarray(y).astype(int)

    gkf = GroupKFold(n_splits=N_SPLITS)

    fold_rows = []
    pred_rows = []
    history_rows = []

    for fold, (tr, va) in enumerate(gkf.split(X_np, y_np, groups)):
        print("\n" + "="*100)
        print("Fold:", fold)

        X_train_raw = X_np[tr]
        X_val_raw = X_np[va]

        y_train = y_np[tr]
        y_val = y_np[va]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
        X_val = scaler.transform(X_val_raw).astype(np.float32)

        model, hist, best_epoch, best_pr = train_one_fold(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            input_dim=X_train.shape[1],
            loss_name=loss_name,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            model_config=model_config,
        )

        val_ds = PairFeatureDataset(X_val, y_val)
        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
        )

        probs = []

        model.eval()
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(DEVICE)
                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()
                probs.extend(prob.tolist())

        metrics = compute_metrics(y_val, probs)
        metrics.update({
            "fold": fold,
            "best_epoch": best_epoch,
            "best_pr_auc_during_training": best_pr,
            "model_name": model_name,
            "loss_name": loss_name,
        })

        fold_rows.append(metrics)

        fold_pred = meta.iloc[va][[
            "pair_id",
            "group_id",
            "enzyme_class",
            "enz_ac",
            "sub_ac",
            "enz_gene",
            "sub_gene",
            "label",
        ]].copy()

        fold_pred["fold"] = fold
        fold_pred["y_true"] = y_val
        fold_pred["prob"] = probs
        fold_pred["model_name"] = model_name
        fold_pred["loss_name"] = loss_name

        pred_rows.append(fold_pred)

        hist["fold"] = fold
        hist["model_name"] = model_name
        hist["loss_name"] = loss_name
        history_rows.append(hist)

        del model
        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        elif DEVICE.type == "mps":
            torch.mps.empty_cache()

    fold_df = pd.DataFrame(fold_rows)
    pred_df = pd.concat(pred_rows, ignore_index=True)
    hist_df = pd.concat(history_rows, ignore_index=True)

    return fold_df, pred_df, hist_df

سلول ۹ — اجرای اولین مدل: NN + BCE

In [ ]:
MODEL_CONFIG_1 = {
    "projection_dim": 1024,
    "block_hidden_dim": 2048,
    "n_blocks": 3,
    "dropout": 0.25,
}

fold_bce, pred_bce, hist_bce = run_nn_cv(
    X_df=X_df,
    y=y,
    groups=groups,
    loss_name="bce",
    model_name="neural_pair_scorer_resmlp_bce",
    model_config=MODEL_CONFIG_1,
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
)

display(fold_bce)

summary_bce = fold_bce.drop(columns=["fold"]).select_dtypes(include=[np.number]).agg(["mean", "std"])
display(summary_bce)

سلول ۱۰ — ذخیره خروجی BCE

In [ ]:
fold_bce.to_csv(
    RESULT_DIR / "nn_bce_fold_metrics.csv",
    index=False,
)

pred_bce.to_csv(
    RESULT_DIR / "nn_bce_fold_predictions.csv",
    index=False,
)

hist_bce.to_csv(
    RESULT_DIR / "nn_bce_training_history.csv",
    index=False,
)

summary_bce.to_csv(
    RESULT_DIR / "nn_bce_summary.csv",
)

print("Saved BCE results.")

سلول ۱۱ — اضافه‌کردن Weighted BCE به تابع train

In [ ]:
def train_one_fold(
    X_train,
    y_train,
    X_val,
    y_val,
    input_dim,
    loss_name="bce",
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
    patience=12,
    model_config=None,
):
    if model_config is None:
        model_config = {}

    train_ds = PairFeatureDataset(X_train, y_train)
    val_ds = PairFeatureDataset(X_val, y_val)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    model = NeuralPairScorer(
        input_dim=input_dim,
        **model_config,
    ).to(DEVICE)

    if loss_name == "bce":
        criterion = nn.BCEWithLogitsLoss()

    elif loss_name == "weighted_bce":
        n_pos = float((y_train == 1).sum())
        n_neg = float((y_train == 0).sum())
        pos_weight_value = n_neg / max(n_pos, 1.0)

        pos_weight = torch.tensor(
            [pos_weight_value],
            dtype=torch.float32,
            device=DEVICE,
        )

        print("pos_weight:", pos_weight_value)

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

    else:
        raise ValueError(loss_name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
    )

    best_pr = -np.inf
    best_state = None
    best_epoch = -1
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_probs = []
        val_true = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)

                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()

                val_probs.extend(prob.tolist())
                val_true.extend(yb.numpy().tolist())

        val_metrics = compute_metrics(val_true, val_probs)
        scheduler.step(val_metrics["pr_auc"])

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            **val_metrics,
        }

        history.append(row)

        if val_metrics["pr_auc"] > best_pr:
            best_pr = val_metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 5 == 0:
            print(
                f"epoch={epoch:03d} "
                f"loss={row['train_loss']:.4f} "
                f"val_pr={row['pr_auc']:.4f} "
                f"val_roc={row['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping at epoch:", epoch)
            break

    model.load_state_dict(best_state)
    model = model.to(DEVICE)
    model.eval()

    return model, pd.DataFrame(history), best_epoch, best_pr

سلول ۱۲ — اجرای Weighted BCE

In [ ]:
fold_wbce, pred_wbce, hist_wbce = run_nn_cv(
    X_df=X_df,
    y=y,
    groups=groups,
    loss_name="weighted_bce",
    model_name="neural_pair_scorer_resmlp_weighted_bce",
    model_config=MODEL_CONFIG_1,
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
)

display(fold_wbce)

summary_wbce = (
    fold_wbce
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(summary_wbce)

سلول ۱۳ — ذخیره خروجی Weighted BCE

In [ ]:
fold_wbce.to_csv(
    RESULT_DIR / "nn_weighted_bce_fold_metrics.csv",
    index=False,
)

pred_wbce.to_csv(
    RESULT_DIR / "nn_weighted_bce_fold_predictions.csv",
    index=False,
)

hist_wbce.to_csv(
    RESULT_DIR / "nn_weighted_bce_training_history.csv",
    index=False,
)

summary_wbce.to_csv(
    RESULT_DIR / "nn_weighted_bce_summary.csv",
)

print("Saved Weighted BCE results.")

سلول ۱۴ — مقایسه BCE و Weighted BCE

In [ ]:
comparison_nn_bce = pd.DataFrame([
    {
        "model": "NN_BCE",
        "roc_auc_mean": fold_bce["roc_auc"].mean(),
        "roc_auc_std": fold_bce["roc_auc"].std(),
        "pr_auc_mean": fold_bce["pr_auc"].mean(),
        "pr_auc_std": fold_bce["pr_auc"].std(),
        "f1_mean": fold_bce["f1"].mean(),
        "f1_std": fold_bce["f1"].std(),
        "accuracy_mean": fold_bce["accuracy"].mean(),
        "precision_mean": fold_bce["precision"].mean(),
        "recall_mean": fold_bce["recall"].mean(),
        "brier_mean": fold_bce["brier"].mean(),
    },
    {
        "model": "NN_Weighted_BCE",
        "roc_auc_mean": fold_wbce["roc_auc"].mean(),
        "roc_auc_std": fold_wbce["roc_auc"].std(),
        "pr_auc_mean": fold_wbce["pr_auc"].mean(),
        "pr_auc_std": fold_wbce["pr_auc"].std(),
        "f1_mean": fold_wbce["f1"].mean(),
        "f1_std": fold_wbce["f1"].std(),
        "accuracy_mean": fold_wbce["accuracy"].mean(),
        "precision_mean": fold_wbce["precision"].mean(),
        "recall_mean": fold_wbce["recall"].mean(),
        "brier_mean": fold_wbce["brier"].mean(),
    },
])

comparison_nn_bce["delta_pr_auc_vs_bce"] = (
    comparison_nn_bce["pr_auc_mean"]
    - comparison_nn_bce.loc[0, "pr_auc_mean"]
)

comparison_nn_bce["delta_roc_auc_vs_bce"] = (
    comparison_nn_bce["roc_auc_mean"]
    - comparison_nn_bce.loc[0, "roc_auc_mean"]
)

display(comparison_nn_bce)

comparison_nn_bce.to_csv(
    RESULT_DIR / "nn_bce_vs_weighted_bce_comparison.csv",
    index=False,
)

سلول ۱۵ — تعریف Focal Loss

In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        targets = targets.float()

        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        prob = torch.sigmoid(logits)

        pt = torch.where(
            targets == 1,
            prob,
            1 - prob
        )

        alpha_t = torch.where(
            targets == 1,
            torch.tensor(self.alpha, device=logits.device),
            torch.tensor(1 - self.alpha, device=logits.device)
        )

        loss = alpha_t * ((1 - pt) ** self.gamma) * bce

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss

سلول ۱۶ — اصلاح تابع train برای Focal

In [ ]:
def train_one_fold(
    X_train,
    y_train,
    X_val,
    y_val,
    input_dim,
    loss_name="bce",
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
    patience=12,
    model_config=None,
):
    if model_config is None:
        model_config = {}

    train_ds = PairFeatureDataset(X_train, y_train)
    val_ds = PairFeatureDataset(X_val, y_val)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    model = NeuralPairScorer(
        input_dim=input_dim,
        **model_config,
    ).to(DEVICE)

    if loss_name == "bce":
        criterion = nn.BCEWithLogitsLoss()

    elif loss_name == "weighted_bce":
        n_pos = float((y_train == 1).sum())
        n_neg = float((y_train == 0).sum())
        pos_weight_value = n_neg / max(n_pos, 1.0)

        pos_weight = torch.tensor(
            [pos_weight_value],
            dtype=torch.float32,
            device=DEVICE,
        )

        print("pos_weight:", pos_weight_value)

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

    elif loss_name == "focal":
        criterion = BinaryFocalLoss(
            alpha=0.35,
            gamma=2.0,
            reduction="mean"
        )

    else:
        raise ValueError(loss_name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
    )

    best_pr = -np.inf
    best_state = None
    best_epoch = -1
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_probs = []
        val_true = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)

                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()

                val_probs.extend(prob.tolist())
                val_true.extend(yb.numpy().tolist())

        val_metrics = compute_metrics(val_true, val_probs)
        scheduler.step(val_metrics["pr_auc"])

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            **val_metrics,
        }

        history.append(row)

        if val_metrics["pr_auc"] > best_pr:
            best_pr = val_metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 5 == 0:
            print(
                f"epoch={epoch:03d} "
                f"loss={row['train_loss']:.4f} "
                f"val_pr={row['pr_auc']:.4f} "
                f"val_roc={row['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping at epoch:", epoch)
            break

    model.load_state_dict(best_state)
    model = model.to(DEVICE)
    model.eval()

    return model, pd.DataFrame(history), best_epoch, best_pr

سلول ۱۷ — اجرای Focal Loss

In [ ]:
fold_focal, pred_focal, hist_focal = run_nn_cv(
    X_df=X_df,
    y=y,
    groups=groups,
    loss_name="focal",
    model_name="neural_pair_scorer_resmlp_focal",
    model_config=MODEL_CONFIG_1,
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
)

display(fold_focal)

summary_focal = (
    fold_focal
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(summary_focal)

سلول ۱۸ — ذخیره Focal

In [ ]:
fold_focal.to_csv(
    RESULT_DIR / "nn_focal_fold_metrics.csv",
    index=False,
)

pred_focal.to_csv(
    RESULT_DIR / "nn_focal_fold_predictions.csv",
    index=False,
)

hist_focal.to_csv(
    RESULT_DIR / "nn_focal_training_history.csv",
    index=False,
)

summary_focal.to_csv(
    RESULT_DIR / "nn_focal_summary.csv",
)

print("Saved Focal Loss results.")

سلول ۱۹ — مقایسه BCE / Weighted BCE / Focal

In [ ]:
comparison_nn_losses = pd.DataFrame([
    {
        "model": "NN_BCE",
        "roc_auc_mean": fold_bce["roc_auc"].mean(),
        "roc_auc_std": fold_bce["roc_auc"].std(),
        "pr_auc_mean": fold_bce["pr_auc"].mean(),
        "pr_auc_std": fold_bce["pr_auc"].std(),
        "f1_mean": fold_bce["f1"].mean(),
        "f1_std": fold_bce["f1"].std(),
        "accuracy_mean": fold_bce["accuracy"].mean(),
        "precision_mean": fold_bce["precision"].mean(),
        "recall_mean": fold_bce["recall"].mean(),
        "brier_mean": fold_bce["brier"].mean(),
    },
    {
        "model": "NN_Weighted_BCE",
        "roc_auc_mean": fold_wbce["roc_auc"].mean(),
        "roc_auc_std": fold_wbce["roc_auc"].std(),
        "pr_auc_mean": fold_wbce["pr_auc"].mean(),
        "pr_auc_std": fold_wbce["pr_auc"].std(),
        "f1_mean": fold_wbce["f1"].mean(),
        "f1_std": fold_wbce["f1"].std(),
        "accuracy_mean": fold_wbce["accuracy"].mean(),
        "precision_mean": fold_wbce["precision"].mean(),
        "recall_mean": fold_wbce["recall"].mean(),
        "brier_mean": fold_wbce["brier"].mean(),
    },
    {
        "model": "NN_Focal",
        "roc_auc_mean": fold_focal["roc_auc"].mean(),
        "roc_auc_std": fold_focal["roc_auc"].std(),
        "pr_auc_mean": fold_focal["pr_auc"].mean(),
        "pr_auc_std": fold_focal["pr_auc"].std(),
        "f1_mean": fold_focal["f1"].mean(),
        "f1_std": fold_focal["f1"].std(),
        "accuracy_mean": fold_focal["accuracy"].mean(),
        "precision_mean": fold_focal["precision"].mean(),
        "recall_mean": fold_focal["recall"].mean(),
        "brier_mean": fold_focal["brier"].mean(),
    },
])

comparison_nn_losses["delta_pr_auc_vs_bce"] = (
    comparison_nn_losses["pr_auc_mean"]
    - comparison_nn_losses.loc[0, "pr_auc_mean"]
)

comparison_nn_losses["delta_roc_auc_vs_bce"] = (
    comparison_nn_losses["roc_auc_mean"]
    - comparison_nn_losses.loc[0, "roc_auc_mean"]
)

display(comparison_nn_losses)

comparison_nn_losses.to_csv(
    RESULT_DIR / "nn_loss_comparison_bce_weighted_focal.csv",
    index=False,
)

سلول ۲۰ — معماری Deep Residual MLP v2

In [ ]:
class ResidualBlockV2(nn.Module):
    def __init__(self, dim, expansion=2, dropout=0.35):
        super().__init__()

        hidden_dim = dim * expansion

        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.net(x)


class NeuralPairScorerV2(nn.Module):
    def __init__(
        self,
        input_dim,
        projection_dim=2048,
        n_blocks=4,
        dropout=0.35,
    ):
        super().__init__()

        self.input = nn.Sequential(
            nn.Linear(input_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.blocks = nn.Sequential(
            *[
                ResidualBlockV2(
                    dim=projection_dim,
                    expansion=2,
                    dropout=dropout,
                )
                for _ in range(n_blocks)
            ]
        )

        self.head = nn.Sequential(
            nn.LayerNorm(projection_dim),
            nn.Linear(projection_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.input(x)
        x = self.blocks(x)
        return self.head(x).squeeze(-1)

سلول ۲۱ — نسخه جدید train که بتواند model class بگیرد

In [ ]:
def train_one_fold_v2(
    X_train,
    y_train,
    X_val,
    y_val,
    input_dim,
    loss_name="focal",
    epochs=100,
    batch_size=128,
    lr=5e-5,
    weight_decay=3e-4,
    patience=15,
    model_config=None,
    model_class=NeuralPairScorerV2,
):
    if model_config is None:
        model_config = {}

    train_ds = PairFeatureDataset(X_train, y_train)
    val_ds = PairFeatureDataset(X_val, y_val)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    model = model_class(
        input_dim=input_dim,
        **model_config,
    ).to(DEVICE)

    if loss_name == "bce":
        criterion = nn.BCEWithLogitsLoss()

    elif loss_name == "weighted_bce":
        n_pos = float((y_train == 1).sum())
        n_neg = float((y_train == 0).sum())
        pos_weight_value = n_neg / max(n_pos, 1.0)

        pos_weight = torch.tensor(
            [pos_weight_value],
            dtype=torch.float32,
            device=DEVICE,
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    elif loss_name == "focal":
        criterion = BinaryFocalLoss(
            alpha=0.35,
            gamma=2.0,
            reduction="mean",
        )

    else:
        raise ValueError(loss_name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    best_pr = -np.inf
    best_state = None
    best_epoch = -1
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )

            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_probs = []
        val_true = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)

                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()

                val_probs.extend(prob.tolist())
                val_true.extend(yb.numpy().tolist())

        val_metrics = compute_metrics(val_true, val_probs)
        scheduler.step(val_metrics["pr_auc"])

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            **val_metrics,
        }

        history.append(row)

        if val_metrics["pr_auc"] > best_pr:
            best_pr = val_metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 5 == 0:
            print(
                f"epoch={epoch:03d} "
                f"loss={row['train_loss']:.4f} "
                f"val_pr={row['pr_auc']:.4f} "
                f"val_roc={row['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping at epoch:", epoch)
            break

    model.load_state_dict(best_state)
    model = model.to(DEVICE)
    model.eval()

    return model, pd.DataFrame(history), best_epoch, best_pr

سلول ۲۲ — CV مخصوص v2

In [ ]:
def run_nn_cv_v2(
    X_df,
    y,
    groups,
    loss_name="focal",
    model_name="neural_pair_scorer_v2_focal",
    model_config=None,
    epochs=100,
    batch_size=128,
    lr=5e-5,
    weight_decay=3e-4,
    model_class=NeuralPairScorerV2,
):
    X_np = X_df.to_numpy(dtype=np.float32)
    y_np = np.asarray(y).astype(int)

    gkf = GroupKFold(n_splits=N_SPLITS)

    fold_rows = []
    pred_rows = []
    history_rows = []

    for fold, (tr, va) in enumerate(gkf.split(X_np, y_np, groups)):
        print("\n" + "=" * 100)
        print("Fold:", fold)

        X_train_raw = X_np[tr]
        X_val_raw = X_np[va]

        y_train = y_np[tr]
        y_val = y_np[va]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
        X_val = scaler.transform(X_val_raw).astype(np.float32)

        model, hist, best_epoch, best_pr = train_one_fold_v2(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            input_dim=X_train.shape[1],
            loss_name=loss_name,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            model_config=model_config,
            model_class=model_class,
        )

        val_ds = PairFeatureDataset(X_val, y_val)
        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
        )

        probs = []

        model.eval()
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(DEVICE)
                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()
                probs.extend(prob.tolist())

        metrics = compute_metrics(y_val, probs)
        metrics.update({
            "fold": fold,
            "best_epoch": best_epoch,
            "best_pr_auc_during_training": best_pr,
            "model_name": model_name,
            "loss_name": loss_name,
        })

        fold_rows.append(metrics)

        fold_pred = meta.iloc[va][[
            "pair_id",
            "group_id",
            "enzyme_class",
            "enz_ac",
            "sub_ac",
            "enz_gene",
            "sub_gene",
            "label",
        ]].copy()

        fold_pred["fold"] = fold
        fold_pred["y_true"] = y_val
        fold_pred["prob"] = probs
        fold_pred["model_name"] = model_name
        fold_pred["loss_name"] = loss_name

        pred_rows.append(fold_pred)

        hist["fold"] = fold
        hist["model_name"] = model_name
        hist["loss_name"] = loss_name
        history_rows.append(hist)

        del model
        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        elif DEVICE.type == "mps":
            torch.mps.empty_cache()

    fold_df = pd.DataFrame(fold_rows)
    pred_df = pd.concat(pred_rows, ignore_index=True)
    hist_df = pd.concat(history_rows, ignore_index=True)

    return fold_df, pred_df, hist_df

سلول ۲۳ — اجرای Deep Residual MLP v2 + Focal

In [ ]:
MODEL_CONFIG_V2 = {
    "projection_dim": 2048,
    "n_blocks": 4,
    "dropout": 0.35,
}

fold_v2_focal, pred_v2_focal, hist_v2_focal = run_nn_cv_v2(
    X_df=X_df,
    y=y,
    groups=groups,
    loss_name="focal",
    model_name="neural_pair_scorer_v2_focal",
    model_config=MODEL_CONFIG_V2,
    epochs=100,
    batch_size=128,
    lr=5e-5,
    weight_decay=3e-4,
    model_class=NeuralPairScorerV2,
)

display(fold_v2_focal)

summary_v2_focal = (
    fold_v2_focal
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(summary_v2_focal)

سلول ۲۴ — ذخیره v2 focal

In [ ]:
fold_v2_focal.to_csv(
    RESULT_DIR / "nn_v2_focal_fold_metrics.csv",
    index=False,
)

pred_v2_focal.to_csv(
    RESULT_DIR / "nn_v2_focal_fold_predictions.csv",
    index=False,
)

hist_v2_focal.to_csv(
    RESULT_DIR / "nn_v2_focal_training_history.csv",
    index=False,
)

summary_v2_focal.to_csv(
    RESULT_DIR / "nn_v2_focal_summary.csv",
)

print("Saved V2 Focal results.")

سلول ۲۵ — مقایسه همه NNها تا اینجا

In [ ]:
comparison_nn_all = pd.DataFrame([
    {
        "model": "NN_BCE",
        "roc_auc_mean": fold_bce["roc_auc"].mean(),
        "roc_auc_std": fold_bce["roc_auc"].std(),
        "pr_auc_mean": fold_bce["pr_auc"].mean(),
        "pr_auc_std": fold_bce["pr_auc"].std(),
        "f1_mean": fold_bce["f1"].mean(),
        "f1_std": fold_bce["f1"].std(),
        "accuracy_mean": fold_bce["accuracy"].mean(),
        "precision_mean": fold_bce["precision"].mean(),
        "recall_mean": fold_bce["recall"].mean(),
        "brier_mean": fold_bce["brier"].mean(),
    },
    {
        "model": "NN_Weighted_BCE",
        "roc_auc_mean": fold_wbce["roc_auc"].mean(),
        "roc_auc_std": fold_wbce["roc_auc"].std(),
        "pr_auc_mean": fold_wbce["pr_auc"].mean(),
        "pr_auc_std": fold_wbce["pr_auc"].std(),
        "f1_mean": fold_wbce["f1"].mean(),
        "f1_std": fold_wbce["f1"].std(),
        "accuracy_mean": fold_wbce["accuracy"].mean(),
        "precision_mean": fold_wbce["precision"].mean(),
        "recall_mean": fold_wbce["recall"].mean(),
        "brier_mean": fold_wbce["brier"].mean(),
    },
    {
        "model": "NN_Focal",
        "roc_auc_mean": fold_focal["roc_auc"].mean(),
        "roc_auc_std": fold_focal["roc_auc"].std(),
        "pr_auc_mean": fold_focal["pr_auc"].mean(),
        "pr_auc_std": fold_focal["pr_auc"].std(),
        "f1_mean": fold_focal["f1"].mean(),
        "f1_std": fold_focal["f1"].std(),
        "accuracy_mean": fold_focal["accuracy"].mean(),
        "precision_mean": fold_focal["precision"].mean(),
        "recall_mean": fold_focal["recall"].mean(),
        "brier_mean": fold_focal["brier"].mean(),
    },
    {
        "model": "NN_V2_Focal",
        "roc_auc_mean": fold_v2_focal["roc_auc"].mean(),
        "roc_auc_std": fold_v2_focal["roc_auc"].std(),
        "pr_auc_mean": fold_v2_focal["pr_auc"].mean(),
        "pr_auc_std": fold_v2_focal["pr_auc"].std(),
        "f1_mean": fold_v2_focal["f1"].mean(),
        "f1_std": fold_v2_focal["f1"].std(),
        "accuracy_mean": fold_v2_focal["accuracy"].mean(),
        "precision_mean": fold_v2_focal["precision"].mean(),
        "recall_mean": fold_v2_focal["recall"].mean(),
        "brier_mean": fold_v2_focal["brier"].mean(),
    },
])

comparison_nn_all["delta_pr_auc_vs_bce"] = (
    comparison_nn_all["pr_auc_mean"]
    - comparison_nn_all.loc[
        comparison_nn_all["model"] == "NN_BCE",
        "pr_auc_mean"
    ].iloc[0]
)

comparison_nn_all["delta_pr_auc_vs_focal"] = (
    comparison_nn_all["pr_auc_mean"]
    - comparison_nn_all.loc[
        comparison_nn_all["model"] == "NN_Focal",
        "pr_auc_mean"
    ].iloc[0]
)

comparison_nn_all = comparison_nn_all.sort_values(
    "pr_auc_mean",
    ascending=False,
)

display(comparison_nn_all)

comparison_nn_all.to_csv(
    RESULT_DIR / "nn_all_models_comparison_until_v2.csv",
    index=False,
)

سلول ۲۶ — Ranking Loss

In [ ]:
class PairwiseRankingLoss(nn.Module):
    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin

    def forward(self, logits, targets, group_ids=None):
        targets = targets.float()
        logits = logits.view(-1)

        pos_scores = logits[targets == 1]
        neg_scores = logits[targets == 0]

        if len(pos_scores) == 0 or len(neg_scores) == 0:
            return torch.tensor(0.0, device=logits.device)

        n = min(len(pos_scores), len(neg_scores), 256)

        pos_idx = torch.randint(0, len(pos_scores), (n,), device=logits.device)
        neg_idx = torch.randint(0, len(neg_scores), (n,), device=logits.device)

        pos_sample = pos_scores[pos_idx]
        neg_sample = neg_scores[neg_idx]

        loss = torch.relu(self.margin - (pos_sample - neg_sample))

        return loss.mean()

سلول ۲۷ — Hybrid Loss

In [ ]:
class HybridBCERankingLoss(nn.Module):
    def __init__(
        self,
        bce_weight=1.0,
        ranking_weight=0.3,
        margin=0.2,
        focal=False,
        alpha=0.35,
        gamma=2.0,
    ):
        super().__init__()

        self.bce_weight = bce_weight
        self.ranking_weight = ranking_weight
        self.focal = focal

        if focal:
            self.classification_loss = BinaryFocalLoss(
                alpha=alpha,
                gamma=gamma,
                reduction="mean",
            )
        else:
            self.classification_loss = nn.BCEWithLogitsLoss()

        self.ranking_loss = PairwiseRankingLoss(
            margin=margin
        )

    def forward(self, logits, targets):
        cls_loss = self.classification_loss(logits, targets)
        rank_loss = self.ranking_loss(logits, targets)

        total_loss = (
            self.bce_weight * cls_loss
            +
            self.ranking_weight * rank_loss
        )

        return total_loss

In [ ]:
def train_one_fold_v2(
    X_train,
    y_train,
    X_val,
    y_val,
    input_dim,
    loss_name="focal",
    epochs=100,
    batch_size=128,
    lr=5e-5,
    weight_decay=3e-4,
    patience=15,
    model_config=None,
    model_class=NeuralPairScorerV2,
):
    if model_config is None:
        model_config = {}

    train_ds = PairFeatureDataset(X_train, y_train)
    val_ds = PairFeatureDataset(X_val, y_val)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    model = model_class(
        input_dim=input_dim,
        **model_config,
    ).to(DEVICE)

    if loss_name == "bce":
        criterion = nn.BCEWithLogitsLoss()

    elif loss_name == "weighted_bce":
        n_pos = float((y_train == 1).sum())
        n_neg = float((y_train == 0).sum())
        pos_weight_value = n_neg / max(n_pos, 1.0)

        pos_weight = torch.tensor(
            [pos_weight_value],
            dtype=torch.float32,
            device=DEVICE,
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    elif loss_name == "focal":
        criterion = BinaryFocalLoss(
            alpha=0.35,
            gamma=2.0,
            reduction="mean",
        )

    elif loss_name == "hybrid_bce_ranking":
        criterion = HybridBCERankingLoss(
            bce_weight=1.0,
            ranking_weight=0.3,
            margin=0.2,
            focal=False,
        )

    elif loss_name == "hybrid_focal_ranking":
        criterion = HybridBCERankingLoss(
            bce_weight=1.0,
            ranking_weight=0.3,
            margin=0.2,
            focal=True,
            alpha=0.35,
            gamma=2.0,
        )

    else:
        raise ValueError(loss_name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    best_pr = -np.inf
    best_state = None
    best_epoch = -1
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )

            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_probs = []
        val_true = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)

                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()

                val_probs.extend(prob.tolist())
                val_true.extend(yb.numpy().tolist())

        val_metrics = compute_metrics(val_true, val_probs)
        scheduler.step(val_metrics["pr_auc"])

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            **val_metrics,
        }

        history.append(row)

        if val_metrics["pr_auc"] > best_pr:
            best_pr = val_metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 5 == 0:
            print(
                f"epoch={epoch:03d} "
                f"loss={row['train_loss']:.4f} "
                f"val_pr={row['pr_auc']:.4f} "
                f"val_roc={row['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping at epoch:", epoch)
            break

    model.load_state_dict(best_state)
    model = model.to(DEVICE)
    model.eval()

    return model, pd.DataFrame(history), best_epoch, best_pr

سلول ۲۹ — اجرای Hybrid Focal + Ranking روی معماری بهتر فعلی

In [ ]:
fold_hybrid_focal_rank, pred_hybrid_focal_rank, hist_hybrid_focal_rank = run_nn_cv_v2(
    X_df=X_df,
    y=y,
    groups=groups,
    loss_name="hybrid_focal_ranking",
    model_name="neural_pair_scorer_resmlp_hybrid_focal_ranking",
    model_config=MODEL_CONFIG_1,
    epochs=80,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-4,
    model_class=NeuralPairScorer,
)

display(fold_hybrid_focal_rank)

summary_hybrid_focal_rank = (
    fold_hybrid_focal_rank
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(summary_hybrid_focal_rank)

سلول ۳۰ — ذخیره Hybrid

In [ ]:
fold_hybrid_focal_rank.to_csv(
    RESULT_DIR / "nn_hybrid_focal_ranking_fold_metrics.csv",
    index=False,
)

pred_hybrid_focal_rank.to_csv(
    RESULT_DIR / "nn_hybrid_focal_ranking_fold_predictions.csv",
    index=False,
)

hist_hybrid_focal_rank.to_csv(
    RESULT_DIR / "nn_hybrid_focal_ranking_training_history.csv",
    index=False,
)

summary_hybrid_focal_rank.to_csv(
    RESULT_DIR / "nn_hybrid_focal_ranking_summary.csv",
)

print("Saved Hybrid Focal Ranking results.")

سلول ۳۱ — مقایسه نهایی NNها

In [ ]:
comparison_nn_all_losses = pd.DataFrame([
    {
        "model": "NN_BCE",
        "roc_auc_mean": fold_bce["roc_auc"].mean(),
        "pr_auc_mean": fold_bce["pr_auc"].mean(),
        "f1_mean": fold_bce["f1"].mean(),
        "accuracy_mean": fold_bce["accuracy"].mean(),
        "precision_mean": fold_bce["precision"].mean(),
        "recall_mean": fold_bce["recall"].mean(),
        "brier_mean": fold_bce["brier"].mean(),
    },
    {
        "model": "NN_Weighted_BCE",
        "roc_auc_mean": fold_wbce["roc_auc"].mean(),
        "pr_auc_mean": fold_wbce["pr_auc"].mean(),
        "f1_mean": fold_wbce["f1"].mean(),
        "accuracy_mean": fold_wbce["accuracy"].mean(),
        "precision_mean": fold_wbce["precision"].mean(),
        "recall_mean": fold_wbce["recall"].mean(),
        "brier_mean": fold_wbce["brier"].mean(),
    },
    {
        "model": "NN_Focal",
        "roc_auc_mean": fold_focal["roc_auc"].mean(),
        "pr_auc_mean": fold_focal["pr_auc"].mean(),
        "f1_mean": fold_focal["f1"].mean(),
        "accuracy_mean": fold_focal["accuracy"].mean(),
        "precision_mean": fold_focal["precision"].mean(),
        "recall_mean": fold_focal["recall"].mean(),
        "brier_mean": fold_focal["brier"].mean(),
    },
    {
        "model": "NN_V2_Focal",
        "roc_auc_mean": fold_v2_focal["roc_auc"].mean(),
        "pr_auc_mean": fold_v2_focal["pr_auc"].mean(),
        "f1_mean": fold_v2_focal["f1"].mean(),
        "accuracy_mean": fold_v2_focal["accuracy"].mean(),
        "precision_mean": fold_v2_focal["precision"].mean(),
        "recall_mean": fold_v2_focal["recall"].mean(),
        "brier_mean": fold_v2_focal["brier"].mean(),
    },
    {
        "model": "NN_Hybrid_Focal_Ranking",
        "roc_auc_mean": fold_hybrid_focal_rank["roc_auc"].mean(),
        "pr_auc_mean": fold_hybrid_focal_rank["pr_auc"].mean(),
        "f1_mean": fold_hybrid_focal_rank["f1"].mean(),
        "accuracy_mean": fold_hybrid_focal_rank["accuracy"].mean(),
        "precision_mean": fold_hybrid_focal_rank["precision"].mean(),
        "recall_mean": fold_hybrid_focal_rank["recall"].mean(),
        "brier_mean": fold_hybrid_focal_rank["brier"].mean(),
    },
])

comparison_nn_all_losses = comparison_nn_all_losses.sort_values(
    "pr_auc_mean",
    ascending=False,
)

display(comparison_nn_all_losses)

comparison_nn_all_losses.to_csv(
    RESULT_DIR / "nn_all_losses_final_comparison.csv",
    index=False,
)

سلول ۳۲ — مسیر فایل‌های لازم

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path(".")

RESULT_DIR = PROJECT_ROOT / "Data_ml" / "neural_pair_scorer_results"
XGB_RESULT_DIR = PROJECT_ROOT / "Data_ml" / "cross_model_fusion_results"

NN_PRED_PATH = RESULT_DIR / "nn_focal_fold_predictions.csv"

# اگر اسم فایل XGB prediction فرق داشت، اول باید ببینیم چه فایل‌هایی داری
for p in sorted(XGB_RESULT_DIR.rglob("*.csv")):
    print(p)

سلول ۳۳ — ساخت predictionهای XGBoost برای بهترین Fusion

In [ ]:
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier

FUSION_DIR = (
    PROJECT_ROOT
    / "Data_ml"
    / "cross_model_fusion_results"
    / "esm650__esm3b__protbert_bfd"
)

X_PATH = FUSION_DIR / "cross_model_fusion.features.parquet"
META_PATH = FUSION_DIR / "meta.csv"

X = pd.read_parquet(X_PATH)
meta_xgb = pd.read_csv(META_PATH, dtype=str, low_memory=False)

y = meta_xgb["label"].astype(int).values
groups = meta_xgb["group_id"].values

def build_xgb_best():
    return XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

gkf = GroupKFold(n_splits=5)

xgb_pred_rows = []
xgb_fold_rows = []

for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
    print("Fold:", fold)
    
    model = build_xgb_best()
    
    Xtr = X.iloc[tr]
    Xte = X.iloc[te]
    
    ytr = y[tr]
    yte = y[te]
    
    model.fit(Xtr, ytr)
    
    prob = model.predict_proba(Xte)[:, 1]
    pred = (prob >= 0.5).astype(int)
    
    xgb_fold_rows.append({
        "fold": fold,
        "roc_auc": roc_auc_score(yte, prob),
        "pr_auc": average_precision_score(yte, prob),
        "f1": f1_score(yte, pred, zero_division=0),
        "accuracy": accuracy_score(yte, pred),
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "brier": brier_score_loss(yte, prob),
    })
    
    fold_pred = meta_xgb.iloc[te][[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "label",
    ]].copy()
    
    fold_pred["fold"] = fold
    fold_pred["y_true"] = yte
    fold_pred["prob_xgb"] = prob
    
    xgb_pred_rows.append(fold_pred)

xgb_fold_metrics = pd.DataFrame(xgb_fold_rows)
xgb_fold_predictions = pd.concat(xgb_pred_rows, ignore_index=True)

display(xgb_fold_metrics)

xgb_summary = xgb_fold_metrics.drop(columns=["fold"]).agg(["mean", "std"])
display(xgb_summary)

xgb_fold_metrics.to_csv(
    RESULT_DIR / "xgb_best_fusion_fold_metrics_for_ensemble.csv",
    index=False,
)

xgb_fold_predictions.to_csv(
    RESULT_DIR / "xgb_best_fusion_fold_predictions_for_ensemble.csv",
    index=False,
)

سلول ۳۴ — خواندن predictionهای NN_Focal و XGB

In [ ]:
nn_pred = pd.read_csv(
    RESULT_DIR / "nn_focal_fold_predictions.csv",
    dtype=str,
    low_memory=False,
)

xgb_pred = pd.read_csv(
    RESULT_DIR / "xgb_best_fusion_fold_predictions_for_ensemble.csv",
    dtype=str,
    low_memory=False,
)

nn_pred["prob_nn"] = nn_pred["prob"].astype(float)
nn_pred["y_true"] = nn_pred["y_true"].astype(int)
nn_pred["fold"] = nn_pred["fold"].astype(int)

xgb_pred["prob_xgb"] = xgb_pred["prob_xgb"].astype(float)
xgb_pred["y_true"] = xgb_pred["y_true"].astype(int)
xgb_pred["fold"] = xgb_pred["fold"].astype(int)

merge_cols = ["pair_id", "fold"]

ens_df = nn_pred.merge(
    xgb_pred[merge_cols + ["prob_xgb"]],
    on=merge_cols,
    how="inner",
)

print("NN pred:", nn_pred.shape)
print("XGB pred:", xgb_pred.shape)
print("Merged:", ens_df.shape)

assert ens_df.shape[0] == nn_pred.shape[0]
assert ens_df["pair_id"].nunique() == ens_df.shape[0]

display(ens_df.head())

سلول ۳۵ — تابع ارزیابی Ensemble

In [ ]:
def evaluate_probs_by_fold(df, prob_col):
    rows = []
    
    for fold, g in df.groupby("fold"):
        y_true = g["y_true"].astype(int).values
        prob = g[prob_col].astype(float).values
        pred = (prob >= 0.5).astype(int)
        
        rows.append({
            "fold": fold,
            "roc_auc": roc_auc_score(y_true, prob),
            "pr_auc": average_precision_score(y_true, prob),
            "f1": f1_score(y_true, pred, zero_division=0),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "brier": brier_score_loss(y_true, prob),
        })
    
    return pd.DataFrame(rows)

سلول ۳۶ — Soft Voting و Weighted Voting

In [ ]:
weights = [
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.4, 0.6),
    (0.3, 0.7),
]

ensemble_results = []
ensemble_fold_tables = []

for wxgb, wnn in weights:
    col = f"ens_xgb{wxgb}_nn{wnn}"
    
    ens_df[col] = (
        wxgb * ens_df["prob_xgb"]
        +
        wnn * ens_df["prob_nn"]
    )
    
    fold_metrics = evaluate_probs_by_fold(ens_df, col)
    fold_metrics["ensemble"] = col
    fold_metrics["w_xgb"] = wxgb
    fold_metrics["w_nn"] = wnn
    
    ensemble_fold_tables.append(fold_metrics)
    
    row = {
        "ensemble": col,
        "w_xgb": wxgb,
        "w_nn": wnn,
    }
    
    for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
        row[f"{m}_mean"] = fold_metrics[m].mean()
        row[f"{m}_std"] = fold_metrics[m].std()
    
    ensemble_results.append(row)

ensemble_weighted_results = pd.DataFrame(ensemble_results).sort_values(
    "pr_auc_mean",
    ascending=False,
)

ensemble_weighted_fold_results = pd.concat(
    ensemble_fold_tables,
    ignore_index=True,
)

display(ensemble_weighted_results)

ensemble_weighted_results.to_csv(
    RESULT_DIR / "ensemble_xgb_nn_weighted_voting_results.csv",
    index=False,
)

ensemble_weighted_fold_results.to_csv(
    RESULT_DIR / "ensemble_xgb_nn_weighted_voting_fold_results.csv",
    index=False,
)

سلول ۳۷ — Stacking با Logistic Regression

In [ ]:
stack_rows = []
stack_pred_rows = []

for test_fold in sorted(ens_df["fold"].unique()):
    train_stack = ens_df[ens_df["fold"] != test_fold].copy()
    test_stack = ens_df[ens_df["fold"] == test_fold].copy()
    
    X_train_stack = train_stack[["prob_xgb", "prob_nn"]].values
    y_train_stack = train_stack["y_true"].astype(int).values
    
    X_test_stack = test_stack[["prob_xgb", "prob_nn"]].values
    y_test_stack = test_stack["y_true"].astype(int).values
    
    meta_model = LogisticRegression(
        solver="liblinear",
        random_state=42,
    )
    
    meta_model.fit(X_train_stack, y_train_stack)
    
    prob_stack = meta_model.predict_proba(X_test_stack)[:, 1]
    pred_stack = (prob_stack >= 0.5).astype(int)
    
    stack_rows.append({
        "fold": test_fold,
        "roc_auc": roc_auc_score(y_test_stack, prob_stack),
        "pr_auc": average_precision_score(y_test_stack, prob_stack),
        "f1": f1_score(y_test_stack, pred_stack, zero_division=0),
        "accuracy": accuracy_score(y_test_stack, pred_stack),
        "precision": precision_score(y_test_stack, pred_stack, zero_division=0),
        "recall": recall_score(y_test_stack, pred_stack, zero_division=0),
        "brier": brier_score_loss(y_test_stack, prob_stack),
        "coef_xgb": meta_model.coef_[0][0],
        "coef_nn": meta_model.coef_[0][1],
        "intercept": meta_model.intercept_[0],
    })
    
    tmp = test_stack[[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "label",
        "fold",
        "y_true",
        "prob_xgb",
        "prob_nn",
    ]].copy()
    
    tmp["prob_stack"] = prob_stack
    stack_pred_rows.append(tmp)

stack_fold_metrics = pd.DataFrame(stack_rows)
stack_predictions = pd.concat(stack_pred_rows, ignore_index=True)

display(stack_fold_metrics)

stack_summary = stack_fold_metrics.drop(columns=["fold"]).select_dtypes(include=[np.number]).agg(["mean", "std"])
display(stack_summary)

stack_fold_metrics.to_csv(
    RESULT_DIR / "ensemble_xgb_nn_stacking_fold_metrics.csv",
    index=False,
)

stack_predictions.to_csv(
    RESULT_DIR / "ensemble_xgb_nn_stacking_predictions.csv",
    index=False,
)

stack_summary.to_csv(
    RESULT_DIR / "ensemble_xgb_nn_stacking_summary.csv",
)

سلول ۳۸ — مقایسه نهایی XGB / NN / Ensemble

In [ ]:
xgb_metrics = pd.read_csv(
    RESULT_DIR / "xgb_best_fusion_fold_metrics_for_ensemble.csv"
)

nn_metrics = pd.read_csv(
    RESULT_DIR / "nn_focal_fold_metrics.csv"
)

final_rows = []

for name, df in [
    ("XGB_best_cross_fusion", xgb_metrics),
    ("NN_Focal", nn_metrics),
]:
    row = {"model": name}
    for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
        row[f"{m}_mean"] = df[m].mean()
        row[f"{m}_std"] = df[m].std()
    final_rows.append(row)

for _, r in ensemble_weighted_results.iterrows():
    final_rows.append({
        "model": r["ensemble"],
        "roc_auc_mean": r["roc_auc_mean"],
        "roc_auc_std": r["roc_auc_std"],
        "pr_auc_mean": r["pr_auc_mean"],
        "pr_auc_std": r["pr_auc_std"],
        "f1_mean": r["f1_mean"],
        "f1_std": r["f1_std"],
        "accuracy_mean": r["accuracy_mean"],
        "precision_mean": r["precision_mean"],
        "recall_mean": r["recall_mean"],
        "brier_mean": r["brier_mean"],
    })

row = {"model": "Stacking_LogReg_XGB_NN"}
for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
    row[f"{m}_mean"] = stack_fold_metrics[m].mean()
    row[f"{m}_std"] = stack_fold_metrics[m].std()
final_rows.append(row)

final_ensemble_comparison = pd.DataFrame(final_rows).sort_values(
    "pr_auc_mean",
    ascending=False,
)

display(final_ensemble_comparison)

final_ensemble_comparison.to_csv(
    RESULT_DIR / "final_xgb_nn_ensemble_comparison.csv",
    index=False,
)